# 🧠 Transformer：通往大模型的“全局视野”

湖北理工学院《人工智能理论》课程资料

📝 **本节核心命题：语义一致性（Semantic Consistency）**
在之前的“文字接龙”体验中，我们发现 FCN 模型如果只能看最后几个词（滑动窗口），它就会陷入“近视眼”的困境。当你让它写一个长句子时，它写着写着就会忘记开头说了什么。

本节我们将通过 **Transformer (Decoder-only)** 展示：
1. **全局注意力机制**：如何像磁铁一样，让结尾的词牢牢锁死开头的逻辑。
2. **自回归生成（Autoregressive Generation）**：真正模拟 ChatGPT “一个字一个字蹦出来”的过程，并观察它是否会产生逻辑幻觉。

---

### 🔍 对垒目标
*   **FCN 组**：窗口大小固定为 8。如果生成了 10 个字，它还能记得主语是“苹果”还是“葡萄”吗？
*   **Transformer 组**：不设窗口。无论中间插入多少废话，看它能否始终如一地输出正确的颜色。

import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

## 🌙 1. 数据准备

我们将生成一个特殊的中文数据集。例如：苹果...红色; 葡萄...紫色。

在长距离干扰下，模型能否通过主语（水果）精准找回对应的宾语（颜色）。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import tiktoken
import random
from IPython.display import clear_output

# ========================================================
# 🚀 设置随机种子以保证结果可复现 (Reproducibility)
# ========================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # 强制使用确定性算法，防止 CUDA 运行时的随机偏差
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
set_seed(1)

# ========================================================
# 👇 设置中文字体，确保图表正常显示汉字
# ========================================================
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# ========================================================
# 🍎 1. 数据准备：强制长距离逻辑锁定
# ========================================================
knowledge_base = {
    "苹果": "红色", "香蕉": "黄色", "葡萄": "紫色",
    "西瓜": "绿色", "橙子": "橙色", "草莓": "红色"
}
fruits = list(knowledge_base.keys())

def generate_long_fact(min_dist=2, max_dist=3):
    f = random.choice(fruits)
    c = knowledge_base[f]
    # 使用丰富的填充词，拉开主语和宾语的距离
    fillers = ["据说", "其实", "非常地", "在那个时候", "根据观察", "看起来", "大概", "根据常识", "众所周知"]
    gap = "".join(random.choices(fillers, k=random.randint(min_dist, max_dist)))
    return f"提示:{f},这种东西{gap}真是{c}。"

# 生成语料
corpus = [generate_long_fact() for _ in range(500)]
enc = tiktoken.get_encoding("cl100k_base")

# BPE 映射 (Method 2)
all_tokens_raw = [enc.encode(s) for s in corpus]
unique_tokens = sorted(list(set([t for seq in all_tokens_raw for t in seq])))
bpe2idx = {t: i for i, t in enumerate(unique_tokens)}
idx2bpe = {i: t for i, t in enumerate(unique_tokens)}
vocab_size = len(unique_tokens)
corpus_mapped = [[bpe2idx[t] for t in seq] for seq in all_tokens_raw]

print(f"✅ 语料生成完毕！词表大小: {vocab_size}")
print(f"🎬 样本示例: {corpus[0]}")

🚜 词嵌入预训练

In [ ]:
# ========================================================
# 虽然我们的终极目标是长文一致性，但底层的词向量逻辑依然需要 Bigram 来锚定
# ========================================================

# 设定隐藏维度为 16
embed_dim = 16 
emb_layer = nn.Embedding(vocab_size, embed_dim)

# 1. 构造 Bigram 训练对 (从长语料中提取每一个相邻词)
bigram_X, bigram_y = [], []
for seq in corpus_mapped:
    for i in range(len(seq) - 1):
        bigram_X.append(seq[i])
        bigram_y.append(seq[i+1])

bigram_X_t = torch.tensor(bigram_X, dtype=torch.long)
bigram_y_t = torch.tensor(bigram_y, dtype=torch.long)

class BigramPreTrainer(nn.Module):
    def __init__(self, emb, v_size):
        super().__init__()
        self.embedding = emb
        self.linear = nn.Linear(16, v_size)
    def forward(self, x):
        return self.linear(self.embedding(x))

pre_trainer = BigramPreTrainer(emb_layer, vocab_size)
pre_opt = optim.Adam(pre_trainer.parameters(), lr=0.01)
criterion_pre = nn.CrossEntropyLoss()

print(f"⏳ 正在预训练局部 Embedding 层...")
for epoch in range(1001):
    logits = pre_trainer(bigram_X_t)
    loss = criterion_pre(logits, bigram_y_t)
    pre_opt.zero_grad(); loss.backward(); pre_opt.step()
    if epoch % 500 == 0:
        print(f"   [Embedding Pretrain] Epoch {epoch:4d} | Loss: {loss.item():.4f}")

# ⚠️ 锁定权重：锁定后将其作为 FCN 和 Transformer 的共享基准
emb_layer.weight.requires_grad = False
print("✅ Embedding 预训练已冻结。现在可以实例化主模型了。")


### 📍 Transformer的核心组件：位置编码 (Positional Encoding)

📝 为什么我们需要位置编码？

在之前的 FCN 方案中，我们是通过 “滑动窗口” 手动定义顺序的。但在 Transformer 中，注意力机制 (Self-Attention) 是全局对称的——无论一个词在句首还是句尾，它与其他词计算关联度的方式完全一样。

这意味着：如果不给模型提供位置信息，模型将无法区分 “张三打了李四” 和 “李四打了张三”。由于我们要进行“文字接龙”，序列的先后顺序（主语在左，宾语在右）是至关重要的逻辑基础。

🧪 正余弦编码的“奥秘”

Transformer 采用了一套固定的、周期的函数来生成坐标。其精妙之处在于：

唯一性：序列中的每一个位置都有一个唯一的向量表示。
相对位置感知：根据三角函数的特性，$PE(pos+k)$ 可以表示为 $PE(pos)$ 的线性变换。这让模型能够轻松学会：“我左边第 3 个词是谁”。

Transformer 内部没有循环结构（RNN）或卷积（CNN），它是并行的。这意味着它“天生”不知道序列的顺序。

我们需要在这个单元格里实现 正余弦位置编码，并画出它的波形图。

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

# ========================================================
# 📍 定义位置编码模块 (与词嵌入向量相加)
# ========================================================

class PositionalEncoding(nn.Module):
    """
    实现经典的 Sine-Cosine 位置编码。
    它的任务是为每个 Token 提供其在序列中的物理“坐标感”。
    """
    def __init__(self, d_model, max_len=128):
        super(PositionalEncoding, self).__init__()
        
        # 1. 创建一个 [max_len, d_model] 的零矩阵，用于存放编码
        pe = torch.zeros(max_len, d_model)
        
        # 2. 生成一个代表位置的序列：[0, 1, 2, ..., max_len-1]
        # unsqueeze(1) 将其变为 [max_len, 1] 方便后续矩阵运算
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        # 3. 计算分母项：其本质是一个按维度递增的波长缩放因子
        # 这里使用 exp + log 的方式是为了数值稳定性
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        
        # 4. 填充编码矩阵：
        # 对偶数维度使用 sin (正弦波)
        pe[:, 0::2] = torch.sin(position * div_term)
        # 对奇数维度使用 cos (余弦波)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # 5. 增加一个批次维度 [1, max_len, d_model] 以适配模型输入
        pe = pe.unsqueeze(0)
        
        # 将其注册为 buffer (缓冲区)，这样它就不属于需要优化的网络权重
        self.register_buffer('pe', pe)

    def forward(self, x):
        """
        前向传播：将位置编码直接“贴”在词向量上
        """
        # 注意：x 的形状是 [batch_size, seq_len, d_model]
        # 我们根据输入的实际长度 seq_len，截取对应的编码并相加
        x = x + self.pe[:, :x.size(1), :]
        return x

# ========================================================
# 📊 位置编码可视化：观察“坐标波形”
# ========================================================
pe_visual = PositionalEncoding(d_model=64, max_len=100)
# 提取编码数据 (移除批次维度)
pe_map = pe_visual.pe[0].numpy()

plt.figure(figsize=(10, 6))
# 绘制热力图展示各维度的数值分布
plt.imshow(pe_map, cmap='RdBu', aspect='auto')
plt.title("Transformer 位置编码 (Positional Encoding) 特征图", fontsize=14)
plt.xlabel("Embedding 维度索引", fontsize=12)
plt.ylabel("序列中的位置索引", fontsize=12)
plt.colorbar(label="编码强度 (Value)")
plt.show()

print("🎓 观察提示：")
print("- 纵轴（位置）的变化表现出明显的周期性（正余弦波动）。")
print("- 不同维度的波长不同，这为模型提供了精密的多分辨率位置感知。")

## 🏰 2. 架构选择：为什么说 Decoding is All You Need?

在 2017 年的开创性论文 *《Attention is All You Need》* 中，研究者最初设计的是由 **编码器 (Encoder)** 和 **解码器 (Decoder)** 组成的双塔架构（用于翻译）。

然而，在处理我们本节课的“文字接龙”或“常识填充”任务时，学术界和工业界（如 GPT 系列、Llama、Qwen 等）已经演进到了 **纯解码器 (Decoder-only)** 架构。原因如下：

*   **1. 因果建模的直极简性**：
    “文字接龙”本质上是 **自回归 (Autoregressive)** 的过程。预测下一个词只需要关注历史信息。Decoder 虽然名字里带个“解”字，但它核心的 **因果掩码 (Causal Masking)** 保证了任何时刻模型都看不到未来的“天机”，这正是接龙任务的完美数学模型。
*   **2. 计算效率的飞跃**：
    相比于双塔架构，Decoder-only 省去了中间的交叉注意力 (Cross-Attention) 层。在相同的内存开销下，我们可以堆叠更多的层数，让模型的“脑容量”更深。
*   **3. 同样的全局视野**：
    即使没有 Encoder，Decoder 块中的 **自注意力 (Self-Attention)** 依然能以 $O(1)$ 的时间距离捕捉序列两端的逻辑。对于“苹果...红色”这样的长跨度关联，它与 Encoder 同样高效。

> 💡 **核心理念**：在本课中，我们不再复刻复杂的翻译模型，而是直接拥抱大模型的 **“GPT 范式”** —— 简洁、高效、具备统治力的 Decoder-only。


In [ ]:
# ========================================================
# 🏗️ 核心模型构建：Transformer 解码器 (Decoder-only)
# 这里的参数量经过校准，旨在与 128 宽度的 FCN 对标
# ========================================================

D_MODEL = 16    # 隐藏维度：每个词在空间中的向量长度
D_FF = 512       # 前馈网络宽度：用于提供复杂的非线性映射，直接决定了模型的“智商”
N_HEAD = 2       # 注意力头数：必须能被 D_MODEL 整除 (16/2=8)
NUM_LAYERS = 3   # 堆叠层数：让特征在层与层之间不断精炼

class TransformerBlock(nn.Module):
    """
    标准 Transformer 块（核心计算单元）：
    它由两条“路”组成，每条路都遵循 [计算 -> 残差相加 -> 归一化] 的循环。
    """
    def __init__(self, d_model, nhead, d_ff):
        super().__init__()
        # 1. 【自注意力机制】：核心“搜索引擎”
        # 这里模型会将向量拆解为 Q (查询), K (键), V (值)
        # Q 与 K 计算相关性得分，然后根据得分去提取相关的 V 信息。
        self.self_attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        
        # 2. 【前馈神经网络】：位置无关的思维模型
        # FFN 在每个序列位置独立运行，相当于为每个 Token 提供了一次深层的“逻辑推理”
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        
        # 3. 【层归一化层】：稳定电压
        # LayerNorm 能够让输入的数据分布保持均值为0、方差为1，防止梯度爆炸或消失
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        # 🧪 第一回路：捕捉词与词之间的联系
        # attn_out: 获取到的上下文信息（比如：看到“苹果”时，注意力会强力拉取“红色”）
        # Residual Connection (x + ...): 将输入直接加到结果上，防止网络太深导致遗忘。
        attn_out, _ = self.self_attn(x, x, x, attn_mask=mask)
        x = self.ln1(x + attn_out) 
        
        # 🧪 第二回路：强化每个位置的特征表达
        # 将上一阶段整合好的上下文特征，通过两层全连接进行非线性变换。
        ff_out = self.ffn(x)
        x = self.ln2(x + ff_out)
        return x

class MiniTransformer(nn.Module):
    """
    微型 GPT 架构：
    它由嵌入层、位置编码、多层 Transformer 块和最终输出层组成。
    """
    def __init__(self, emb, d_model=D_MODEL, nhead=N_HEAD, num_layers=NUM_LAYERS):
        super().__init__()
        # 1. 共享词嵌入层 (来自 Method 2 预训练)
        self.embedding = emb
        # 2. 位置编码：手动补全 Transformers 对位置的视觉盲点
        self.pos_encoder = PositionalEncoding(d_model, max_len=128)
        # 3. 重复堆叠核心块：多层堆叠能让模型理解更高阶的抽象逻辑
        self.layers = nn.ModuleList([TransformerBlock(d_model, nhead, D_FF) for _ in range(num_layers)])
        # 4. 语言预测头：将 16 维的高级特征映射回词表大小，输出概率分布
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        # [A] 准备阶段
        x = self.embedding(x)
        x = self.pos_encoder(x)
        
        # [B] 安全屏障：生成因果掩码 (Causal Mask)
        sz = x.size(1)
        mask = torch.triu(torch.ones(sz, sz) * float('-inf'), diagonal=1).to(x.device)
        
        # [C] 深度演进：通过多层注意力处理
        for layer in self.layers:
            x = layer(x, mask=mask)
        
        # 🧪 关键修正：不再只取最后一位特征，而是返回全序列 [Batch, Seq_Len, Vocab_Size]
        # 这也是 GPT 训练的核心：让每一位都学会预测下一位
        return self.fc_out(x)

class BaselineFCN(nn.Module):
    """
    基准模型：3 层深度隐藏层 (128 单元)
    严格遵循线性层 + 激活函数的经典堆叠架构。
    """
    def __init__(self, emb, v_size, window_size=4, h_dim=128):
        super().__init__()
        self.embedding = emb
        self.window_size = window_size
        
        # 💡 输入层计算：window_size * d_model = 8 * 16 = 128
        self.net = nn.Sequential(
            nn.Linear(window_size * D_MODEL, h_dim), # 输入层 (128 -> 128)
            nn.ReLU(),                   
            nn.Linear(h_dim, h_dim),                 # 隐藏层 1 (128 -> 128)
            nn.ReLU(),
            nn.Linear(h_dim, h_dim),                 # 隐藏层 2 (128 -> 128)
            nn.ReLU(),
            nn.Linear(h_dim, v_size)                 # 输出层 (128 -> vocab_size)
        )
    def forward(self, x):
        # 1. 裁剪最后 window_size 个 Token (视野受限)
        x_win = x[:, -self.window_size:]
        # 2. 查表并展平：[Batch, 8, 16] -> [Batch, 128]
        e = self.embedding(x_win).reshape(x_win.size(0), -1)
        # 3. 经过 3 层全连接网络
        return self.net(e)

# ========================================================
# 📏 参数对垒与最终确认
# ========================================================
model_tf = MiniTransformer(emb_layer)
# FCN 结构保持不变
model_fcn = BaselineFCN(emb_layer, v_size=vocab_size)

def count_training_params(model):
    # 排除了被冻结的共享词向量参数
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("-" * 55)
print(f"🤖 Transformer (3层, d_ff=512) 有效训练参数量: {count_training_params(model_tf):,}")
print(f"🧱 FCN (3层, 128宽) 有效训练参数量:         {count_training_params(model_fcn):,}")
print("-" * 55)


## 🏗️ 3. 对比测试

我们分别训练FCN和Transformer，并让这两个模型在同一个“大海捞针”任务中进行对垒。

我们在 FCN 中设定一个固定的滑动窗口（例如 8），如果水果和颜色的距离超过了 8，FCN 理论上就会“看不见”主语，从而容易失败。

当然，自定义的迷你Transformer仅进行了简单的微调，结果也不一定好到哪去。

In [ ]:
# ========================================================
# 📦 数据重新对齐：适配长距离逻辑 (Confirmed Edition)
# ========================================================

# --- Transformer：全序列因果对齐 ---
# 计算当前语料中最长的序列
max_seq_len = max(len(s) for s in corpus_mapped)

X_tf_list, Y_tf_list = [], []

for seq in corpus_mapped:
    # 填充到统一长度，使用 0 作为 [PAD]
    pad_len = max_seq_len - len(seq)
    full_seq = seq + [0] * pad_len
    
    # 💡 核心 GPT 逻辑：X 为 [0...N-1], Y 为 [1...N]
    # 即：看到这一步，预测下一步
    X_tf_list.append(full_seq[:-1])
    Y_tf_list.append(full_seq[1:])

X_tf_tensor = torch.tensor(X_tf_list, dtype=torch.long)
Y_tf_tensor = torch.tensor(Y_tf_list, dtype=torch.long)

# --- FCN：滑动视野对齐 (窗口固定为 8) ---
WS = 4
X_fcn_list, y_fcn_list = [], []

for seq in corpus_mapped:
    if len(seq) <= WS: continue
    for i in range(len(seq) - WS):
        X_fcn_list.append(seq[i : i + WS]) # 输入 8 个词
        y_fcn_list.append(seq[i + WS])     # 预测第 9 个词

X_fcn_tensor = torch.tensor(X_fcn_list, dtype=torch.long)
y_fcn_tensor = torch.tensor(y_fcn_list, dtype=torch.long)

print(f"📐 长度对齐完成：")
print(f"最大序列跨度: {max_seq_len} 个 Token")
print(f"Transformer 样本总量: {len(X_tf_list)} 条完整序列")
print(f"FCN 滑动窗口样本量: {len(X_fcn_list)} 个局部视野")

🚀 Transformer 接龙训练：强化因果关联

这个部分的脚本需要确保损失函数（Loss）能正确忽略我们在 Padding 位里填的 0，这样模型才不会去尝试学习“预测空白”。

In [ ]:
# ========================================================
# 🚀 Transformer 训练：全局语义锁定训练
# ========================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_tf.to(device)
opt_tf = optim.Adam(model_tf.parameters(), lr=0.001)

# 💡 关键：ignore_index=0 确保模型不学习 Padding (填充) 部分
criterion_tf = nn.CrossEntropyLoss(ignore_index=0) 

X_tf_train = X_tf_tensor.to(device)
Y_tf_train = Y_tf_tensor.to(device)

loss_history_tf = []
epochs_tf = 300

print(f"⚡ Transformer 语义训练启动 (Device: {device})...")

for epoch in range(epochs_tf):
    model_tf.train()
    indices = torch.randperm(X_tf_train.size(0))
    epoch_l = 0
    
    for i in range(0, X_tf_train.size(0), 32):
        idx = indices[i : i + 32]
        X_b, Y_b = X_tf_train[idx], Y_tf_train[idx]
        
        # 前向传播 (此时模型已修正为返回全序列 Logits)
        logits = model_tf(X_b) 
        
        # 展平进行交叉熵计算：[Batch*SeqLen, Vocab] 对标 [Batch*SeqLen]
        loss = criterion_tf(logits.reshape(-1, vocab_size), Y_b.reshape(-1))
        
        opt_tf.zero_grad(); loss.backward(); opt_tf.step()
        epoch_l += loss.item()

    if (epoch + 1) % 30 == 0 or epoch == 0:
        avg_l = epoch_l / (X_tf_train.size(0)/32)
        loss_history_tf.append(avg_l)
        clear_output(wait=True)
        plt.figure(figsize=(10, 4))
        plt.plot(loss_history_tf, marker='o', color='royalblue', label='Sequence Loss')
        plt.title(f"Transformer 训练曲线 (Epoch {epoch+1})")
        plt.xlabel("监控点"); plt.ylabel("Loss")
        plt.grid(alpha=0.3); plt.legend(); plt.show()

print(f"✅ Transformer 训练完成！模型已具备维持长文本一致性的基础。")

训练基准模型FCN

In [ ]:
# ========================================================
# 🥊 FCN 训练：局部视野的局限性学习
# ========================================================
model_fcn.to(device)
opt_fcn = optim.Adam(model_fcn.parameters(), lr=0.001)
criterion_fcn = nn.CrossEntropyLoss() # FCN 样本通常不包含填充，直接交叉熵即可

X_fcn_train = X_fcn_tensor.to(device)
y_fcn_train = y_fcn_tensor.to(device)

print(f"⏳ 正在训练 FCN 基准模型 (窗口大小: {WS})...")

for epoch in range(300):
    model_fcn.train()
    indices = torch.randperm(X_fcn_train.size(0))
    epoch_l = 0
    
    # 💡 FCN 的样本量通常很大（因为是滑出来的），我们用 64 作为 Batch
    for i in range(0, X_fcn_train.size(0), 64):
        idx = indices[i : i + 64]
        X_b, y_b = X_fcn_train[idx], y_fcn_train[idx]
        
        logits = model_fcn(X_b)
        loss = criterion_fcn(logits, y_b)
        
        opt_fcn.zero_grad(); loss.backward(); opt_fcn.step()
        epoch_l += loss.item()

    if (epoch + 1) % 30 == 0:
        print(f"   [FCN] Epoch {epoch+1:4d} | Loss: {epoch_l/(X_fcn_train.size(0)/64):.4f}")

print(f"✅ FCN 训练完成！")

自回归推理：模拟 ChatGPT 的逻辑

In [ ]:
# ========================================================
# 🚀 自回归生成函数 (Autoregressive Generation)
# ========================================================

def chat_generate(model, prompt, max_gen=30, model_type='transformer'):
    model.eval()
    device = next(model.parameters()).device
    
    # 1. 编码输入并转换为本地 ID 列表
    raw_tokens = enc.encode(prompt)
    input_ids = [bpe2idx[t] for t in raw_tokens if t in bpe2idx]
    
    # 获取终止符 ID
    eos_token = enc.encode("。")[0]
    eos_id = bpe2idx.get(eos_token, -1)
    for _ in range(max_gen):
        # 将当前已生成的 ID 列表转为 Tensor [1, SeqLen]
        curr_tensor = torch.tensor([input_ids]).to(device)
        
        with torch.no_grad():
            if model_type == 'transformer':
                # Transformer 处理整个变长序列
                logits = model(curr_tensor) # 输出 [1, SeqLen, Vocab]
                next_token_logits = logits[0, -1, :]
            else:
                # FCN 严格限制视野为 WS (8)
                seq_len = curr_tensor.size(1)
                if seq_len < WS:
                    # 如果当前字数不足 8 个，左侧补 0 (Padding)
                    padding = torch.zeros((1, WS - seq_len), dtype=torch.long).to(device)
                    win_tensor = torch.cat([padding, curr_tensor], dim=1)
                else:
                    # 如果字数超过 8 个，截取最后 8 个
                    win_tensor = curr_tensor[:, -WS:]
                
                # 此时 win_tensor 保证了它是 [1, 8] 形状
                next_token_logits = model(win_tensor)[0]
        
            # 2. 预测下一个 Token
            next_id = torch.argmax(next_token_logits).item()
            input_ids.append(next_id)
            
            # 3. 检查是否触碰终止符
            if next_id == eos_id:
                break
                
    # 解码并返回结果
    return enc.decode([idx2bpe[i] for i in input_ids])

In [ ]:
# ========================================================
# 🍎 语义一致性验证测试 (The Consistency Test)
# ========================================================

print("🧪 正在启动语义一致性测试...")
print("-" * 60)

results_table = []

for fruit in fruits:
    prompt = f"提示:{fruit}"
    correct_color = knowledge_base[fruit]
    
    # 让两个模型各显神通
    res_tf = chat_generate(model_tf, prompt, model_type='transformer')
    res_fcn = chat_generate(model_fcn, prompt, model_type='fcn')
    
    # 验证逻辑：检查生成的最后 2 个字符（或搜索整个生成文本）是否包含正确颜色
    tf_check = "✅ 合格" if correct_color in res_tf else "❌ 逻辑崩溃"
    fcn_check = "✅ 合格" if correct_color in res_fcn else "❌ 逻辑崩溃"
    
    results_table.append([fruit, correct_color, tf_check, fcn_check])

# 打印美化后的结果表
import pandas as pd
df = pd.DataFrame(results_table, columns=["测试主语", "应得颜色", "Transformer (全局)", "FCN (滑动窗口)"])
display(df)

# ========================================================
# 🎓 深度总结：为什么 FCN 会“逻辑崩溃”？
# ========================================================
print("\n📝 教学诊断报告：")
print("1. FCN 的记忆盲区：FCN 的输入被阉割到了最后 8 个 Token。")
print(f"   当模型自己生成了第 9 个‘废话’字时，开头的‘{fruits[0]}’就从它的神经网络里消失了！")
print("   没有了主语，FCN 只能根据局部的‘这种东西...真是’来胡乱匹配一个颜色特征。")
print("2. Transformer 的锁定效应：由于每个 Token 都能看到之前的‘{fruits[0]}’，")
print("   即使后面生成上万字，最后输出颜色的那一刻，它依然能与主语完成‘跨时空对齐’。")


In [ ]:
# ========================================================
# 🔎 模型生成过程全景检查（诊断专用）
# ========================================================

def model_diagnosis(test_fruits):
    print(f"{'测试主语':<10} | {'预期结果':<8} | {'Transformer 生成路径 (Global)':<40} | {'FCN 生成路径 (Sliding)'}")
    print("-" * 120)
    
    for f in test_fruits:
        prompt = f"提示:{f},"
        ground_truth = knowledge_base[f]
        
        # 获取生成全文本
        raw_tf = chat_generate(model_tf, prompt, model_type='transformer', max_gen=25)
        raw_fcn = chat_generate(model_fcn, prompt, model_type='fcn', max_gen=25)
        
        # 截断展示，方便阅读
        display_tf = (raw_tf[:50] + "..") if len(raw_tf) > 50 else raw_tf
        display_fcn = (raw_fcn[:50] + "..") if len(raw_fcn) > 50 else raw_fcn
        
        print(f"{f:<10} | {ground_truth:<8} | {display_tf:<40} | {display_fcn}")

# 选取几个代表性的水果进行检查
model_diagnosis(["苹果", "香蕉", "葡萄"])
